# 🚀 OCR TownHub — CHẠY SERVICE (GPU) + Public URL

Chạy service 3 engine (VietOCR + PaddleOCR + Gemini) trên **GPU**, mở **URL công khai** để demo / nối backend .NET.

**Trước tiên:** Runtime → Change runtime type → **T4 GPU**.

**Thứ tự chạy:**
1. Cell **1** (condacolab) → Colab **tự khởi động lại** (báo *Your session crashed* — bình thường). ĐỢI nó xong.
2. Sau restart, chạy lần lượt Cell **2 → 3 → 4 → 5 → 6**. **KHÔNG** chạy lại Cell 1.

> Vì sao condacolab? Colab hiện là Python 3.12, `paddlepaddle-gpu 2.6.1` chỉ có bản cho Python ≤3.11. condacolab hạ về Python 3.10 để paddle chạy GPU.

### Cell 1 — Ép Python 3.10 (condacolab). Chạy XONG sẽ tự restart.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # tự restart kernel; sau đó chạy tiếp từ Cell 2

### Cell 2 — Cài thư viện (bản GPU). ~4-6 phút.

In [ ]:
import sys; print('Python', sys.version.split()[0])   # condacolab thường cho 3.10/3.11
!nvidia-smi -L || echo '⚠️ Chưa bật GPU: Runtime → Change runtime type → T4 GPU'
!apt-get -qq install -y poppler-utils libgl1 libglib2.0-0 > /dev/null
# setuptools<81 để còn 'pkg_resources' (gdown của vietocr cần); bản >=81 đã gỡ bỏ.
!pip install -q --force-reinstall 'setuptools<81'
!pip install -q fastapi 'uvicorn[standard]' pydantic requests pdf2image pillow google-generativeai
!pip install -q torch torchvision easyocr vietocr
!pip install -q 'paddlepaddle-gpu==2.6.1' 'paddleocr==2.7.3'
# numpy 1.x + opencv CÀI SAU CÙNG để ghim ABI (tránh 'numpy.core.multiarray failed to import').
!pip install -q --force-reinstall --no-deps 'numpy==1.26.4' 'opencv-python-headless==4.9.0.80'
print('✅ Cài xong')

### Cell 3 — Lấy code + mount Drive + trỏ weight

In [ ]:
import os
REPO = 'https://github.com/thuongerikdev/TownHub'
if not os.path.exists('/content/townhub'):
    !git clone --depth 1 $REPO /content/townhub
else:
    !cd /content/townhub && git pull -q
os.chdir('/content/townhub/ocr-service'); print('cwd:', os.getcwd())

from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/townhub_ocr'

os.environ['VIETOCR_WEIGHTS'] = f'{DRIVE}/weights/vietocr_invoice.pth'
os.environ['PADDLE_REC_DIR']  = f'{DRIVE}/inference/rec_vi'
os.environ['PADDLE_DET_DIR']  = f'{DRIVE}/inference/det_vi'
os.environ['PADDLE_REC_DICT'] = f'{DRIVE}/dict_vi.txt'
os.environ['OCR_USE_GPU'] = '1'                       # ép dùng GPU
os.environ['OCRKEY']     = 'doan-ocr-2026'            # khớp OCR_API_KEY phía .NET
# os.environ['GEMINIKEY'] = 'AIza...'                 # chỉ cần nếu dùng engine gemini

for k in ['VIETOCR_WEIGHTS','PADDLE_REC_DIR','PADDLE_DET_DIR','PADDLE_REC_DICT']:
    print(('✅' if os.path.exists(os.environ[k]) else '❌ THIẾU'), k, '=', os.environ[k])

### Cell 4 — Kiểm tra GPU đã nhận trong torch/paddle

In [ ]:
import torch, paddle
print('torch CUDA :', torch.cuda.is_available())
print('paddle CUDA:', paddle.is_compiled_with_cuda(), '| gpu count:', paddle.device.cuda.device_count())

### Cell 5 — Mở tunnel công khai (cloudflared). Copy URL để show/nối .NET.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess, re
p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:7860','--no-autoupdate'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('\n\n🌐 URL CÔNG KHAI =', m.group(0))
        print('   → /health để kiểm tra, /extract để gọi OCR')
        print('   → .NET: OCR_SERVICE_URL =', m.group(0), '| OCR_API_KEY = doan-ocr-2026')
        break

### Cell 6 — Chạy service *(giữ cell này chạy suốt buổi demo)*
Lần gọi đầu mỗi engine sẽ nạp model (vài giây). Mở URL ở Cell 5 + `/health` để kiểm tra.

In [ ]:
!python app.py